# original-eat-rest-preserved (eat-rest-v1) -- real simulation test, seeds 1-10

Runs the `original-eat-rest-preserved` external candidate ("loose_appetite250", the
unmodified original rest/eat controller that `original-eat-rest-overcrowding-v4` branched
from -- no pawn overcrowding overlay) through `external/candidates/run_candidate.py`, the
same headless game loop as `training/batch_runner.py`'s `_run_one`:
`SimulationCore(seed=seed, starting_predators=0)`, `MAX_SIM_TIME=3000`, one `sim.step()`
per tick, candidate's own `make_policy()`/`decide_all()` called every step. This is a real
full game to extinction or the 3000s time limit each run, not a decision-check suite.

This notebook actually **executes** the candidate rather than loading a precomputed CSV.
Same pattern as `test_astra.ipynb`, pointed at the `original-eat-rest-preserved` candidate
via `run_candidate.py --candidate original-eat-rest-preserved` instead of the v4 default.

**Cluster setup:** the first cells clone/refresh the repo the same way `survival_baseline.ipynb`
does. Before running on the cluster, create a git-ignored `.env` file in the kernel's
starting directory containing `GITHUB_TOKEN=<a token with repo read access>`.


In [13]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}


/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [14]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 16 (delta 5), reused 16 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 334.37 KiB | 7.43 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   ec9554d..a9ccffb  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating ec9554d..a9ccffb
Fast-forward
 .../survival-simulator/agent_server.py             |   17 +-
 .../original-eat-rest-preserved/README.md          |   11 +
 .../original-source-snapshot.zip                   |  Bin 0 -> 159825 bytes
 .../origina

In [15]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

In [19]:
import subprocess
import sys
import time
SEEDS = list(range(1, 20))
SCRIPT = "external/candidates/run_candidate.py"
CANDIDATE = "original-eat-rest-preserved"
OUT_CSV = "results/EXTERNAL_" + CANDIDATE + "_seeds1-10.csv"

cmd = [sys.executable, SCRIPT, "--seeds", *map(str, SEEDS), "--out", OUT_CSV, "--candidate", CANDIDATE]
print("running:", " ".join(cmd))

t0 = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.perf_counter() - t0

print(f"finished in {elapsed / 60:.1f} min, exit code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"run_candidate.py failed with exit code {proc.returncode}")


running: /opt/conda/bin/python external/candidates/run_candidate.py --seeds 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 --out results/EXTERNAL_original-eat-rest-preserved_seeds1-10.csv --candidate original-eat-rest-preserved
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/p

In [20]:
import pandas as pd

df = pd.read_csv(OUT_CSV)
df.loc[df["agent"] == "original_eat_rest_preserved", "agent"] = "v1"
df = df.sort_values("seed").reset_index(drop=True)
df


,agent,seed,score,extinction_time,end_reason,wall_clock_sec
0,v1,1,1904.8107,1887.8,extinction,167.62
1,v1,2,1621.3882,1618.9,extinction,182.47
2,v1,3,1830.6998,1831.2,extinction,193.52
3,v1,4,791.1374,833.2,extinction,66.40
4,v1,5,1268.1955,1242.1,extinction,122.73
5,v1,6,1643.8758,1589.9,extinction,136.25
6,v1,7,1837.6815,1808.9,extinction,179.92
7,v1,8,1748.5609,1726.5,extinction,169.33
8,v1,9,1450.3508,1427.7,extinction,101.42
9,v1,10,1627.0639,1632.2,extinction,130.07


In [18]:
df["score"].agg(["mean", "median", "std", "min", "max"])

mean      1572.376450
median    1635.469850
std        334.485841
min        791.137400
max       1904.810700
Name: score, dtype: float64